# 🚀 NEXUS Stock AI — Streamlit Dashboard Launcher

This notebook provides a **100% self-contained background launcher** for the **NEXUS Stock AI Streamlit Dashboard**.

### 🌟 Key Enhancements & Fixes:
1. **Modern Streamlit Compatibility:** Replaced deprecated `use_column_width=True` with `width="stretch"` to eliminate `TypeError`.
2. **Safe HTML/CSS Rendering:** Custom elements (badges, metric cards, FinBERT sentiment tags) render natively via `st.html` / clean dedented markdown.
3. **True Background Daemon Execution:** Streamlit and the tunnel run with `nohup ... &`. The notebook cell runs a non-blocking health check and exits in 5–8 seconds without freezing.
4. **WebSocket & JS-Chunk Reliability:** Replaced unstable Localtunnel with **Cloudflare Tunnel (`cloudflared`)** and direct **VS Code Port Forwarding (Port 8501)**, completely eliminating broken widgets and password friction.

### ⚙️ Step 1: Install Dependencies & Mount Google Drive

In [21]:
# 1. Install prerequisites
!pip install -q streamlit plotly pyarrow xgboost

import os
import sys
import time
import shutil
import zipfile

# 2. Mount Google Drive
try:
    from google.colab import drive
    print("Mounting Google Drive at /content/drive...")
    drive.mount("/content/drive")
    print("✓ Google Drive mounted successfully.")
except Exception as e:
    print(f"Drive mount note: {e}")

# 3. Unpack nexus_data_backup.zip if available
zip_candidates = [
    "/content/drive/MyDrive/NEXUS_Stock_AI/nexus_data_backup.zip",
    "./nexus_data_backup.zip",
    "/content/nexus_data_backup.zip"
]
unpacked = False
for zpath in zip_candidates:
    if os.path.exists(zpath):
        print(f"Unpacking backup archive from: {zpath}...")
        with zipfile.ZipFile(zpath, 'r') as zf:
            zf.extractall(".")
        print("✓ Unpacked backup archive.")
        unpacked = True
        break

if not unpacked:
    print("ℹ️ No backup zip found yet. Checking Drive folders...")

# 4. Link Google Drive data & models directories if present
drive_data = "/content/drive/MyDrive/NEXUS_Stock_AI/data"
if os.path.exists(drive_data) and not os.path.exists("./data"):
    try:
        os.symlink(drive_data, "./data")
        print("✓ Linked Google Drive data folder to ./data")
    except Exception:
        pass

drive_models = "/content/drive/MyDrive/NEXUS_Stock_AI/models"
if os.path.exists(drive_models) and not os.path.exists("./models"):
    try:
        os.symlink(drive_models, "./models")
        print("✓ Linked Google Drive models folder to ./models")
    except Exception:
        pass


Mounting Google Drive at /content/drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted successfully.
Unpacking backup archive from: /content/drive/MyDrive/NEXUS_Stock_AI/nexus_data_backup.zip...
✓ Unpacked backup archive.


### 📝 Step 2: Auto-Scaffold `streamlit_app/app.py` & `src/inference/predict.py`
Ensures all latest bug-free application files and inference engines are written to disk and synced with Drive.

In [22]:
import os
import base64
import shutil

# Create directories
os.makedirs("streamlit_app", exist_ok=True)
os.makedirs("src/inference", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

# 1. Write src/inference/__init__.py
with open("src/inference/__init__.py", "w", encoding="utf-8") as f:
    f.write('from .predict import NexusInferenceEngine\n__all__ = ["NexusInferenceEngine"]\n')

# 2. Write src/inference/predict.py
predict_b64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKTkVYVVMgU3RvY2sgQUkg4oCUIFN0YW5kYWxvbmUgSW5mZXJlbmNlIEVuZ2luZSAoUGhhc2VzIDIz4oCTMjgpClByb3ZpZGVzIHByb2R1Y3Rpb24tcmVhZHkgaW5mZXJlbmNlIGFuZCBsb2NhbCBmYWN0b3IgZXhwbGFpbmFiaWxpdHkgZm9yIG5leHQtZGF5IGRpcmVjdGlvbi4KU2VhbWxlc3NseSBzdXBwb3J0cyBib3RoIENhbGlicmF0ZWRDbGFzc2lmaWVyQ1Ygam9ibGliIHBpcGVsaW5lcyBhbmQgbmF0aXZlIFhHQm9vc3QgSlNPTiBtb2RlbHMuCiIiIgoKaW1wb3J0IG9zCmltcG9ydCBqc29uCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0LCBEaWN0LCBBbnksIFVuaW9uLCBPcHRpb25hbAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgeGdib29zdCBhcyB4Z2IKCgpjbGFzcyBOZXh1c0luZmVyZW5jZUVuZ2luZToKICAgICIiIgogICAgUHJvZHVjdGlvbiBpbmZlcmVuY2UgZW5naW5lIGZvciBORVhVUyBTdG9jayBBSS4KICAgIExvYWRzIENhbGlicmF0ZWRDbGFzc2lmaWVyQ1Ygam9ibGliIG1vZGVscyBvciBuYXRpdmUgWEdCb29zdCBKU09OIGFydGlmYWN0cywKICAgIGFuZCBydW5zIHZhbGlkYXRlZCwgZXhwbGFpbmFibGUgcHJlZGljdGlvbnMuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBtb2RlbF9wYXRoOiBPcHRpb25hbFtzdHJdID0gTm9uZSwKICAgICAgICBzY2hlbWFfcGF0aDogc3RyID0gIi4vbW9kZWxzL2ZlYXR1cmVfc2NoZW1hLmpzb24iCiAgICApOgogICAgICAgICMgMS4gUmVzb2x2ZSBtb2RlbCBwYXRoIHdpdGggcHJpb3JpdGl6ZWQgZmFsbGJhY2sgY2FuZGlkYXRlcwogICAgICAgIHJlc29sdmVkX21vZGVsX3BhdGggPSBOb25lCiAgICAgICAgY2FuZGlkYXRlcyA9IFtdCiAgICAgICAgaWYgbW9kZWxfcGF0aDoKICAgICAgICAgICAgY2FuZGlkYXRlcy5leHRlbmQoWwogICAgICAgICAgICAgICAgbW9kZWxfcGF0aCwKICAgICAgICAgICAgICAgIG9zLnBhdGguam9pbigiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9ORVhVU19TdG9ja19BSS9tb2RlbHMiLCBvcy5wYXRoLmJhc2VuYW1lKG1vZGVsX3BhdGgpKQogICAgICAgICAgICBdKQogICAgICAgIGNhbmRpZGF0ZXMuZXh0ZW5kKFsKICAgICAgICAgICAgIi4vbW9kZWxzL3hnYl9kaXJlY3Rpb25fY2FsaWJyYXRlZC5qb2JsaWIiLAogICAgICAgICAgICAiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9ORVhVU19TdG9ja19BSS9tb2RlbHMveGdiX2RpcmVjdGlvbl9jYWxpYnJhdGVkLmpvYmxpYiIsCiAgICAgICAgICAgICIuL21vZGVscy94Z2JfZGlyZWN0aW9uLmpzb24iLAogICAgICAgICAgICAiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9ORVhVU19TdG9ja19BSS9tb2RlbHMveGdiX2RpcmVjdGlvbi5qc29uIgogICAgICAgIF0pCgogICAgICAgIGZvciBwIGluIGNhbmRpZGF0ZXM6CiAgICAgICAgICAgIGlmIHAgYW5kIG9zLnBhdGguZXhpc3RzKHApOgogICAgICAgICAgICAgICAgcmVzb2x2ZWRfbW9kZWxfcGF0aCA9IHAKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgICAgIGlmIHJlc29sdmVkX21vZGVsX3BhdGggaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNb2RlbCBmaWxlIG5vdCBmb3VuZC4gQ2hlY2tlZCBjYW5kaWRhdGUgbG9jYXRpb25zOiB7Y2FuZGlkYXRlc30iKQoKICAgICAgICAjIDIuIFJlc29sdmUgZmVhdHVyZSBzY2hlbWEgcGF0aAogICAgICAgIHJlc29sdmVkX3NjaGVtYV9wYXRoID0gTm9uZQogICAgICAgIHNjaGVtYV9jYW5kaWRhdGVzID0gWwogICAgICAgICAgICBzY2hlbWFfcGF0aCwKICAgICAgICAgICAgb3MucGF0aC5qb2luKCIvY29udGVudC9kcml2ZS9NeURyaXZlL05FWFVTX1N0b2NrX0FJL21vZGVscyIsIG9zLnBhdGguYmFzZW5hbWUoc2NoZW1hX3BhdGgpKSwKICAgICAgICAgICAgIi4vbW9kZWxzL2ZlYXR1cmVfc2NoZW1hLmpzb24iCiAgICAgICAgXQogICAgICAgIGZvciBzcCBpbiBzY2hlbWFfY2FuZGlkYXRlczoKICAgICAgICAgICAgaWYgc3AgYW5kIG9zLnBhdGguZXhpc3RzKHNwKToKICAgICAgICAgICAgICAgIHJlc29sdmVkX3NjaGVtYV9wYXRoID0gc3AKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgICAgIGlmIHJlc29sdmVkX3NjaGVtYV9wYXRoIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiU2NoZW1hIGZpbGUgbm90IGZvdW5kLiBDaGVja2VkIGNhbmRpZGF0ZSBsb2NhdGlvbnM6IHtzY2hlbWFfY2FuZGlkYXRlc30iKQoKICAgICAgICB3aXRoIG9wZW4ocmVzb2x2ZWRfc2NoZW1hX3BhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgc2NoZW1hX2RhdGEgPSBqc29uLmxvYWQoZikKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShzY2hlbWFfZGF0YSwgZGljdCkgYW5kICJmZWF0dXJlcyIgaW4gc2NoZW1hX2RhdGE6CiAgICAgICAgICAgICAgICBzZWxmLmZlYXR1cmVfbmFtZXMgPSBzY2hlbWFfZGF0YVsiZmVhdHVyZXMiXQogICAgICAgICAgICBlbGlmIGlzaW5zdGFuY2Uoc2NoZW1hX2RhdGEsIGxpc3QpOgogICAgICAgICAgICAgICAgc2VsZi5mZWF0dXJlX25hbWVzID0gc2NoZW1hX2RhdGEKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuZXhwZWN0ZWQgc2NoZW1hIGZvcm1hdC4gRXhwZWN0ZWQgbGlzdCBvciBkaWN0IHdpdGggJ2ZlYXR1cmVzJyBrZXkuIikKCiAgICAgICAgc2VsZi5tb2RlbF9wYXRoID0gcmVzb2x2ZWRfbW9kZWxfcGF0aAogICAgICAgIHNlbGYuc2NoZW1hX3BhdGggPSByZXNvbHZlZF9zY2hlbWFfcGF0aAoKICAgICAgICAjIDMuIExvYWQgbW9kZWwgYmFzZWQgb24gZmlsZSBmb3JtYXQKICAgICAgICBpZiBzZWxmLm1vZGVsX3BhdGguZW5kc3dpdGgoIi5qb2JsaWIiKToKICAgICAgICAgICAgaW1wb3J0IGpvYmxpYgogICAgICAgICAgICBzZWxmLm1vZGVsID0gam9ibGliLmxvYWQoc2VsZi5tb2RlbF9wYXRoKQogICAgICAgICAgICBzZWxmLmJvb3N0ZXIgPSBOb25lCgogICAgICAgICAgICAjIEV4dHJhY3QgdW5kZXJseWluZyBYR0Jvb3N0IGJvb3N0ZXIgZnJvbSBDYWxpYnJhdGVkQ2xhc3NpZmllckNWCiAgICAgICAgICAgIGlmIGhhc2F0dHIoc2VsZi5tb2RlbCwgImNhbGlicmF0ZWRfY2xhc3NpZmllcnNfIikgYW5kIGxlbihzZWxmLm1vZGVsLmNhbGlicmF0ZWRfY2xhc3NpZmllcnNfKSA+IDA6CiAgICAgICAgICAgICAgICBmaXJzdF9jYWwgPSBzZWxmLm1vZGVsLmNhbGlicmF0ZWRfY2xhc3NpZmllcnNfWzBdCiAgICAgICAgICAgICAgICBiYXNlX2VzdCA9IGdldGF0dHIoZmlyc3RfY2FsLCAiZXN0aW1hdG9yIiwgZ2V0YXR0cihmaXJzdF9jYWwsICJiYXNlX2VzdGltYXRvciIsIE5vbmUpKQogICAgICAgICAgICAgICAgaWYgaGFzYXR0cihiYXNlX2VzdCwgImdldF9ib29zdGVyIik6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5ib29zdGVyID0gYmFzZV9lc3QuZ2V0X2Jvb3N0ZXIoKQogICAgICAgICAgICAgICAgZWxpZiBoYXNhdHRyKGJhc2VfZXN0LCAiYm9vc3RlciIpOgogICAgICAgICAgICAgICAgICAgIHNlbGYuYm9vc3RlciA9IGJhc2VfZXN0LmJvb3N0ZXIKICAgICAgICAgICAgZWxpZiBoYXNhdHRyKHNlbGYubW9kZWwsICJnZXRfYm9vc3RlciIpOgogICAgICAgICAgICAgICAgc2VsZi5ib29zdGVyID0gc2VsZi5tb2RlbC5nZXRfYm9vc3RlcigpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5tb2RlbCA9IHhnYi5YR0JDbGFzc2lmaWVyKCkKICAgICAgICAgICAgc2VsZi5tb2RlbC5sb2FkX21vZGVsKHNlbGYubW9kZWxfcGF0aCkKICAgICAgICAgICAgc2VsZi5ib29zdGVyID0gc2VsZi5tb2RlbC5nZXRfYm9vc3RlcigpCgogICAgZGVmIHByZWRpY3QoCiAgICAgICAgc2VsZiwKICAgICAgICBpbnB1dF9mZWF0dXJlczogcGQuRGF0YUZyYW1lLAogICAgICAgIHRvcF9rX2ZhY3RvcnM6IGludCA9IDMKICAgICkgLT4gTGlzdFtEaWN0W3N0ciwgQW55XV06CiAgICAgICAgIiIiCiAgICAgICAgUnVucyBiYXRjaCBkaXJlY3Rpb25hbCBwcmVkaWN0aW9uIGFuZCBTSEFQIGZhY3RvciBhdHRyaWJ1dGlvbi4KICAgICAgICAiIiIKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShpbnB1dF9mZWF0dXJlcywgcGQuRGF0YUZyYW1lKToKICAgICAgICAgICAgcmFpc2UgVHlwZUVycm9yKCJpbnB1dF9mZWF0dXJlcyBtdXN0IGJlIGEgcGFuZGFzIERhdGFGcmFtZS4iKQoKICAgICAgICBtaXNzaW5nX2NvbHMgPSBbYyBmb3IgYyBpbiBzZWxmLmZlYXR1cmVfbmFtZXMgaWYgYyBub3QgaW4gaW5wdXRfZmVhdHVyZXMuY29sdW1uc10KICAgICAgICBpZiBtaXNzaW5nX2NvbHM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbnB1dCBmZWF0dXJlcyBtaXNzaW5nIHJlcXVpcmVkIHNjaGVtYSBjb2x1bW5zICh7bGVuKG1pc3NpbmdfY29scyl9KToge21pc3NpbmdfY29sc30iKQoKICAgICAgICBYID0gaW5wdXRfZmVhdHVyZXNbc2VsZi5mZWF0dXJlX25hbWVzXS5jb3B5KCkKCiAgICAgICAgaWYgWC5pc25hKCkuYW55KCkuYW55KCk6CiAgICAgICAgICAgIG5hbl9jb2xzID0gWC5jb2x1bW5zW1guaXNuYSgpLmFueSgpXS50b2xpc3QoKQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW5wdXQgZmVhdHVyZXMgY29udGFpbiBOYU4gdmFsdWVzIGluIGNvbHVtbnM6IHtuYW5fY29sc30iKQoKICAgICAgICBwcm9icyA9IHNlbGYubW9kZWwucHJlZGljdF9wcm9iYShYKQogICAgICAgIHByZWRzID0gKHByb2JzWzosIDFdID49IDAuNSkuYXN0eXBlKGludCkKCiAgICAgICAgaWYgc2VsZi5ib29zdGVyIGlzIG5vdCBOb25lOgogICAgICAgICAgICBkbWF0cml4ID0geGdiLkRNYXRyaXgoWCwgZmVhdHVyZV9uYW1lcz1zZWxmLmZlYXR1cmVfbmFtZXMpCiAgICAgICAgICAgIGNvbnRyaWJzID0gc2VsZi5ib29zdGVyLnByZWRpY3QoZG1hdHJpeCwgcHJlZF9jb250cmlicz1UcnVlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNvbnRyaWJzID0gbnAuemVyb3MoKGxlbihYKSwgbGVuKHNlbGYuZmVhdHVyZV9uYW1lcykgKyAxKSkKCiAgICAgICAgcmVzdWx0cyA9IFtdCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKGlucHV0X2ZlYXR1cmVzKSk6CiAgICAgICAgICAgIHJvd19yYXcgPSBpbnB1dF9mZWF0dXJlcy5pbG9jW2ldCiAgICAgICAgICAgIHRpY2tlciA9IHN0cihyb3dfcmF3LmdldCgic3RvY2tfc3ltYm9sIiwgcm93X3Jhdy5nZXQoInRpY2tlciIsICJVTktOT1dOIikpKQogICAgICAgICAgICAKICAgICAgICAgICAgdHJhZGVfZGF0ZV92YWwgPSByb3dfcmF3LmdldCgiZGF0ZSIsIHJvd19yYXcuZ2V0KCJ0cmFkZV9kYXRlIiwgIlVOS05PV04iKSkKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZSh0cmFkZV9kYXRlX3ZhbCwgKHBkLlRpbWVzdGFtcCwgbnAuZGF0ZXRpbWU2NCkpOgogICAgICAgICAgICAgICAgdHJhZGVfZGF0ZSA9IHBkLnRvX2RhdGV0aW1lKHRyYWRlX2RhdGVfdmFsKS5zdHJmdGltZSgiJVktJW0tJWQiKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgdHJhZGVfZGF0ZSA9IHN0cih0cmFkZV9kYXRlX3ZhbCkKCiAgICAgICAgICAgIHVwX3Byb2IgPSBmbG9hdChyb3VuZChwcm9ic1tpLCAxXSwgNCkpCiAgICAgICAgICAgIGRvd25fcHJvYiA9IGZsb2F0KHJvdW5kKHByb2JzW2ksIDBdLCA0KSkKICAgICAgICAgICAgcHJlZGljdGlvbl9sYWJlbCA9ICJVUCIgaWYgcHJlZHNbaV0gPT0gMSBlbHNlICJET1dOIgoKICAgICAgICAgICAgcm93X2NvbnRyaWJzID0gY29udHJpYnNbaSwgOi0xXQogICAgICAgICAgICBmYWN0b3JfbGlzdCA9IFtdCiAgICAgICAgICAgIGZvciBmZWF0X25hbWUsIGltcGFjdF92YWwgaW4gemlwKHNlbGYuZmVhdHVyZV9uYW1lcywgcm93X2NvbnRyaWJzKToKICAgICAgICAgICAgICAgIGZhY3Rvcl9saXN0LmFwcGVuZCh7CiAgICAgICAgICAgICAgICAgICAgImZlYXR1cmUiOiBmZWF0X25hbWUsCiAgICAgICAgICAgICAgICAgICAgImRpcmVjdGlvbiI6ICJVUCIgaWYgaW1wYWN0X3ZhbCA+IDAgZWxzZSAiRE9XTiIsCiAgICAgICAgICAgICAgICAgICAgImltcGFjdCI6IGZsb2F0KHJvdW5kKGFicyhpbXBhY3RfdmFsKSwgNCkpCiAgICAgICAgICAgICAgICB9KQoKICAgICAgICAgICAgZmFjdG9yX2xpc3Quc29ydChrZXk9bGFtYmRhIHg6IHhbImltcGFjdCJdLCByZXZlcnNlPVRydWUpCiAgICAgICAgICAgIHRvcF9mYWN0b3JzID0gZmFjdG9yX2xpc3RbOnRvcF9rX2ZhY3RvcnNdCgogICAgICAgICAgICByZXN1bHRzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAidGlja2VyIjogdGlja2VyLAogICAgICAgICAgICAgICAgInRyYWRlX2RhdGUiOiB0cmFkZV9kYXRlLAogICAgICAgICAgICAgICAgInByZWRpY3Rpb24iOiBwcmVkaWN0aW9uX2xhYmVsLAogICAgICAgICAgICAgICAgInVwX3Byb2JhYmlsaXR5IjogdXBfcHJvYiwKICAgICAgICAgICAgICAgICJkb3duX3Byb2JhYmlsaXR5IjogZG93bl9wcm9iLAogICAgICAgICAgICAgICAgInRvcF9mYWN0b3JzIjogdG9wX2ZhY3RvcnMKICAgICAgICAgICAgfSkKCiAgICAgICAgcmV0dXJuIHJlc3VsdHMK"
with open("src/inference/predict.py", "wb") as f:
    f.write(base64.b64decode(predict_b64))
print(f"✓ Verified src/inference/predict.py ({os.path.getsize('src/inference/predict.py')} bytes)")

# 3. Write streamlit_app/app.py
app_b64 = "..."
with open("streamlit_app/app.py", "wb") as f:
    f.write(base64.b64decode(app_b64))
print(f"✓ Verified streamlit_app/app.py ({os.path.getsize('streamlit_app/app.py')} bytes)")

# 4. Sync code to Google Drive if connected
GDRIVE_BASE = "/content/drive/MyDrive/NEXUS_Stock_AI"
if os.path.exists(GDRIVE_BASE):
    os.makedirs(os.path.join(GDRIVE_BASE, "streamlit_app"), exist_ok=True)
    os.makedirs(os.path.join(GDRIVE_BASE, "src", "inference"), exist_ok=True)
    shutil.copy2("streamlit_app/app.py", os.path.join(GDRIVE_BASE, "streamlit_app", "app.py"))
    shutil.copy2("src/inference/predict.py", os.path.join(GDRIVE_BASE, "src", "inference", "predict.py"))
    shutil.copy2("src/inference/__init__.py", os.path.join(GDRIVE_BASE, "src", "inference", "__init__.py"))
    print("✓ Synced code to Google Drive.")


✓ Verified src/inference/predict.py (6621 bytes)
✓ Verified streamlit_app/app.py (18869 bytes)
✓ Synced code to Google Drive.


### 🌐 Step 3: Launch Streamlit Background Daemon & Establish Connection

This cell:
1. Cleans up any stale port 8501 processes.
2. Starts Streamlit in the **background** with WebSocket and CORS proxy flags.
3. Verifies that Streamlit is actively serving HTTP 200.
4. Launches **Cloudflare Tunnel (`cloudflared`)** in the background and extracts the public URL.
5. **Exits in 5–8 seconds without freezing your notebook!**

In [7]:
import os
import time
import re

# 1. Clean up stale processes
!fuser -k 8501/tcp 2>/dev/null || true
!pkill -f streamlit 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
!pkill -f localtunnel 2>/dev/null || true
time.sleep(1)

# 2. Launch Streamlit server in the background with proxy flags
print("Starting Streamlit server on port 8501 in background...")
!nohup streamlit run streamlit_app/app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false --browser.gatherUsageStats false > streamlit.log 2>&1 &

# 3. Health check loop (wait up to 12s)
server_up = False
for i in range(12):
    time.sleep(1)
    if os.system("curl -s http://127.0.0.1:8501 > /dev/null") == 0:
        server_up = True
        break

if server_up:
    print("✓ Streamlit server is UP and responding on http://127.0.0.1:8501!")
else:
    print("⚠️ Streamlit server did not respond within 12s. Check streamlit.log:")
    !cat streamlit.log

# 4. Install & Launch Cloudflare Tunnel daemon
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Installing Cloudflare Tunnel (cloudflared)... ")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print("✓ cloudflared installed.")

print("Establishing Cloudflare Tunnel in background...")
!nohup /usr/local/bin/cloudflared tunnel --url http://127.0.0.1:8501 > tunnel.log 2>&1 &

# 5. Extract public URL from tunnel.log (poll up to 10s)
tunnel_url = None
for _ in range(10):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
            matches = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if matches:
                tunnel_url = matches[-1]
                break

print("\n" + "=" * 72)
print("🚀 NEXUS STOCK AI DASHBOARD IS READY!")
print("=" * 72)

if tunnel_url:
    print(f"\n🌐 PUBLIC CLOUDFLARE URL (No Password Required, Fast & Stable):\n👉 {tunnel_url}\n")
else:
    print("\n⚠️ Tunnel initialization took longer than expected. Run `!cat tunnel.log` to inspect.")

print("-" * 72)
print("⚡ ZERO-LATENCY ACCESS: VS CODE PORT FORWARDING (RECOMMENDED)")
print("1. In VS Code, open the 'Ports' panel (Ctrl+Shift+P -> 'View: Toggle Ports').")
print("2. Click 'Forward a Port' and enter: 8501")
print("3. Open http://localhost:8501 directly in your browser!")
print("=" * 72)


 41512^C
^C
^C
Starting Streamlit server on port 8501 in background...
✓ Streamlit server is UP and responding on http://127.0.0.1:8501!
Establishing Cloudflare Tunnel in background...

🚀 NEXUS STOCK AI DASHBOARD IS READY!

🌐 PUBLIC CLOUDFLARE URL (No Password Required, Fast & Stable):
👉 https://dow-analyze-cornell-impaired.trycloudflare.com

------------------------------------------------------------------------
⚡ ZERO-LATENCY ACCESS: VS CODE PORT FORWARDING (RECOMMENDED)
1. In VS Code, open the 'Ports' panel (Ctrl+Shift+P -> 'View: Toggle Ports').
2. Click 'Forward a Port' and enter: 8501
3. Open http://localhost:8501 directly in your browser!


### 📦 Step 4: Refresh Project Backup Archive & Maintenance
Packages all dashboard, model, and dataset files into `nexus_data_backup.zip`.

In [ ]:
target_zip = "/content/nexus_data_backup.zip" if os.path.exists("/content") else "./nexus_data_backup.zip"
print(f"Refreshing backup archive: {target_zip}...")

with zipfile.ZipFile(target_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for folder in ["./streamlit_app", "./src", "./models", "./data"]:
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for file in files:
                    if not file.endswith(".pyc") and "__pycache__" not in root:
                        fpath = os.path.join(root, file)
                        arcname = os.path.relpath(fpath, ".")
                        zipf.write(fpath, arcname)

zip_size = os.path.getsize(target_zip) / (1024 * 1024)
print(f"✓ Backup archive updated successfully ({zip_size:.2f} MB).")

gdrive_backup = "/content/drive/MyDrive/NEXUS_Stock_AI/nexus_data_backup.zip"
if os.path.exists("/content/drive/MyDrive/NEXUS_Stock_AI"):
    shutil.copy2(target_zip, gdrive_backup)
    print(f"✓ Synced updated archive to Google Drive: {gdrive_backup}")


In [ ]:
# Useful commands to check logs or stop the background server:
# !cat streamlit.log
# !cat tunnel.log
# !fuser -k 8501/tcp; pkill -f streamlit; pkill -f cloudflared
